# ReCAP study — Colab first run + reduced pilot

Repo: `https://github.com/laraib-b/ReCAP-study`

**This notebook is a file copy.** If the GitHub notebook changes, re-download/open the new version from the repo. Pushing code does **not** auto-update an already-open Colab tab.

Sections:
1. Clone / refresh code
2. Install deps
3. Synthetic smoke (fast)
4. Reduced **DOW30** real-data pilot (slower; not paper-scale)

## 1) Clone or refresh the repo

If `/content/ReCAP` already exists from an older run, this cell runs `git pull`.

In [ ]:
REPO_URL = "https://github.com/laraib-b/ReCAP-study.git"
REPO_DIR = "/content/ReCAP"
BRANCH = "main"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("cwd:", os.getcwd())
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2) Install dependencies

Core stack only (no full FinRL required for this pilot — tickers are vendored in `study/market_tickers.py`).

In [ ]:
# Pin versions that play nicer with current Google Colab base images.
%pip install -q "numpy>=1.24,<2.5" "pandas==2.2.3" "torch" "gymnasium>=0.29" "stable-baselines3>=2.0" "yfinance>=0.2" "requests>=2.31" pytest
import os
os.environ["YF_DISABLE_CURL_CFFI"] = "1"
print("deps ready; downloads use plain requests fallback")

## 3) Synthetic smoke (optional)

Skip this section if Colab is tight on RAM — go straight to section 4 after install.

In [ ]:
%run study/smoke_run.py

## 4) Micro real-data pilot (Colab-safe)

Free Colab often **crashes (OOM)** on full DOW30 + long history. This section uses profile **`micro`**:
- 8 tickers only
- Dates ~2019–2021
- Tiny PPO budgets

Still **not** paper-scale. Output JSON under `outputs/pilots/`.

If this succeeds, we can scale up later.

In [ ]:
import os
import torch

os.environ["YF_DISABLE_CURL_CFFI"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| cuda:", torch.cuda.is_available())

# MICRO profile — avoids free-Colab RAM kills from full DOW30 + 2008-2025
!python study/reduced_pilot.py --profile micro --dataset DOW30 --device {device} --seed 0 --disable-local-cache

In [ ]:
from pathlib import Path
import json

pilot_dir = Path("outputs/pilots")
files = sorted(pilot_dir.glob("*.json")) if pilot_dir.exists() else []
print("pilot files:", [p.name for p in files])
if files:
    latest = files[-1]
    payload = json.loads(latest.read_text())
    print("latest:", latest.name)
    print("metrics:", payload.get("metrics"))
    print("warning:", payload.get("warning"))

## After this works

1. Download the JSON from `outputs/pilots/` (Colab Files panel) for your notes.
2. Tell Cursor the run finished — next we add causal eval / FA calibration on top of this path.
3. Do **not** treat these CR/SR/MDD numbers as paper replication.